# Week 13: Machine Learning vs. Regression Demo
## Bias-Variance Tradeoff and 4-Model Comparison Using REIT Factor Data

---

**What you will learn:**

- Why in-sample fit is a misleading measure of model quality — and how the train/test split reveals the truth
- How OLS, Ridge, Lasso, and Random Forest differ in how they balance bias and variance
- How to diagnose overfitting by comparing training R² vs. test R²
- How feature importance differs between linear models and tree-based models — and what that means for interpretation

**How to use this notebook:**
Run cells from top to bottom. Read the interpretation notes after each chart — they explain what to look for and why it matters for real investment decisions.

**Dataset:** Monthly REIT factor returns from the `factors_master_long_only.csv` file — 470 monthly observations (December 1986 – January 2026). We use `vwtret` (value-weighted REIT return) as our prediction target.

In [ ]:
# =============================================================================
# SECTION 0: SETUP AND IMPORTS
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Use a clean, student-friendly plot style
plt.style.use('seaborn-v0_8-whitegrid')

# Color palette — one consistent color per model throughout the notebook
MODEL_COLORS = {
    'OLS':          'steelblue',
    'Ridge':        'darkorange',
    'Lasso':        'purple',
    'RandomForest': 'darkgreen'
}

# Random seed for reproducibility
RANDOM_SEED = 42

print("Imports loaded successfully.")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

---
## Section 1: Load the Data

In [ ]:
# =============================================================================
# SECTION 1, CELL 1: Load CSV and parse dates
# =============================================================================

# Path — adjust if running from a different directory
import os
import sys

# Search upward from the notebook location to find the Data/ folder
from pathlib import Path

notebook_dir = Path(os.getcwd())
data_path = None
for parent in [notebook_dir] + list(notebook_dir.parents):
    candidate = parent / 'Data' / 'factors_master_long_only.csv'
    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find Data/factors_master_long_only.csv. "
        "Make sure you are running this notebook from inside the repo."
    )

print(f"Found data at: {data_path}")

# Load the CSV
df_raw = pd.read_csv(data_path)

# Parse dates: 'date' column contains YYYYMM integers (e.g. 198612)
df_raw['date'] = pd.to_datetime(df_raw['date'].astype(str), format='%Y%m')

print(f"\nDataset shape: {df_raw.shape}  ({df_raw.shape[0]} rows, {df_raw.shape[1]} columns)")
print(f"Date range: {df_raw['date'].min().strftime('%b %Y')} to {df_raw['date'].max().strftime('%b %Y')}")
print()
df_raw.head(3)

**What does this data represent?**

Each row is one calendar month. The column `vwtret` is the **value-weighted return** on a portfolio of REIT (Real Estate Investment Trust) stocks — essentially, how much a dollar invested in the broad REIT market grew (or shrank) that month, in percentage points. The other columns are **factor returns** from academic asset-pricing research: size premium, value premium, momentum, quality, and low-volatility. We are going to try to predict next month's REIT return using these factors — which is exactly the kind of forecasting problem where machine learning is often proposed as a tool.

In [ ]:
# =============================================================================
# SECTION 1, CELL 3: Summary statistics for vwtret
# =============================================================================

stats_vwtret = df_raw['vwtret'].describe()

print("Summary statistics for vwtret (value-weighted REIT return, monthly %)")
print("-" * 55)
print(f"  Count  : {stats_vwtret['count']:.0f} months")
print(f"  Mean   : {stats_vwtret['mean']:.3f} %")
print(f"  Std dev: {stats_vwtret['std']:.3f} %")
print(f"  Min    : {stats_vwtret['min']:.3f} %")
print(f"  Max    : {stats_vwtret['max']:.3f} %")

---
## Section 2: The Core Question — Inference vs. Prediction

Before building any model, we need to decide **what question we're actually asking.**

| | OLS / Econometric Models | Machine Learning |
|---|---|---|
| **Question** | *Why* did returns change? | *What* will returns be? |
| **Output** | β coefficients with p-values | Minimized prediction error (RMSE / R²) |
| **Strength** | Causal interpretation, auditable | Captures non-linear patterns, scales to many features |
| **Weakness** | May underfit complex data | Black-box; can overfit |
| **Use case** | Academic research, regulatory reporting | Screening, portfolio sorting |

**Neither approach is universally better.** The right choice depends on your decision. If you want to tell a story about *which* factor drives REIT returns, OLS gives you a coefficient you can defend. If you want the lowest forecast error on next month's return, ML may (or may not) outperform OLS — and the train/test split will tell you which.

In [ ]:
# =============================================================================
# SECTION 2, CELL 2: Annual volatility of vwtret
# This chart motivates why prediction is hard — returns vary dramatically by era.
# =============================================================================

df_raw['year'] = df_raw['date'].dt.year

annual_vol = df_raw.groupby('year')['vwtret'].std().reset_index()
annual_vol.columns = ['year', 'monthly_std']

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(annual_vol['year'], annual_vol['monthly_std'],
       color='steelblue', alpha=0.75, edgecolor='white')

# Annotate some notable years
notable = {2008: '2008\nGFC', 2020: '2020\nCOVID'}
for yr, label in notable.items():
    row = annual_vol[annual_vol['year'] == yr]
    if not row.empty:
        ax.annotate(label, xy=(yr, row['monthly_std'].values[0]),
                    xytext=(yr, row['monthly_std'].values[0] + 0.5),
                    ha='center', fontsize=9, color='darkred',
                    arrowprops=dict(arrowstyle='->', color='darkred'))

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Monthly Return Std Dev (%)', fontsize=11)
ax.set_title('Annual Volatility of REIT Returns — Why Prediction is Hard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The GFC (2008) and COVID (2020) episodes spike the volatility dramatically.")
print("Any model trained on calm periods will struggle in turbulent ones — and vice versa.")

---
## Section 3: Train-Test Split

### Why hold out a test set?

If we fit a model and then measure its accuracy on the **same data** we used to fit it, we are asking: "How well does this model describe the past?" That's a different question from "How well will this model predict the future?"

**Data leakage** happens when information from the test set influences model training. The result is an overly optimistic in-sample R² that collapses when the model sees new data. The classic ML mistake is to report training accuracy as if it were prediction accuracy.

**The fix:** Before fitting anything, split the data into:

- **Training set (70%)** — the model sees this data and learns from it
- **Test set (30%)** — held out, never touched during training; used only to evaluate final performance

The test set is our "out-of-sample exam." A high test R² means the model genuinely learned a pattern that generalizes beyond the training period.

In [ ]:
# =============================================================================
# SECTION 3, CELL 2: Select features, drop NaN rows, define X and y
# =============================================================================

# All numeric columns except the date column and the target vwtret
all_numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in all_numeric_cols if col not in ['vwtret', 'year']]

print("Feature columns to be used as predictors:")
for col in feature_cols:
    n_missing = df_raw[col].isna().sum()
    print(f"  {col:<25}  ({n_missing} missing values)")

# Drop rows with any NaN in features OR target
df_clean = df_raw[feature_cols + ['vwtret']].dropna().copy()
print(f"\nRows before dropping NaN: {len(df_raw)}")
print(f"Rows after  dropping NaN: {len(df_clean)}  (used for modeling)")

# Define outcome and feature matrix explicitly
y_return = df_clean['vwtret']           # Outcome: monthly value-weighted REIT return
X_features = df_clean[feature_cols]    # Predictors: factor returns

# 70/30 split — shuffle=True because months are i.i.d. enough for this demo
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_return,
    test_size=0.30,
    random_state=RANDOM_SEED
)

print(f"\nTrain set size: {len(X_train)} months  ({len(X_train)/len(df_clean)*100:.0f}%)")
print(f"Test  set size: {len(X_test)} months  ({len(X_test)/len(df_clean)*100:.0f}%)")

# Standardize features (important for Ridge and Lasso so penalties are on the same scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit only on training data
X_test_scaled  = scaler.transform(X_test)       # apply same scaling to test data

print("\nFeatures standardized (mean=0, std=1) — required for fair Ridge/Lasso penalty comparison.")

**Interpretation Checkpoint — Train-Test Split**

- We have split the 470 months into roughly 330 training months and 140 test months.
- The scaler was fit *only* on training data and then applied to test data. Fitting the scaler on all data would be a subtle form of data leakage — the test set would influence the standardization.
- **The test set is our "out-of-sample exam." Models should never see test data during fitting.**

---
## Section 4: OLS Baseline

In [ ]:
# =============================================================================
# SECTION 4, CELL 1: Fit OLS and compute R² on train and test
# =============================================================================

from sklearn.linear_model import LinearRegression

# Fit OLS explicitly on the training data
model_ols = LinearRegression()
model_ols.fit(X_train_scaled, y_train)

# Predictions
y_pred_train_ols = model_ols.predict(X_train_scaled)
y_pred_test_ols  = model_ols.predict(X_test_scaled)

# R² — what fraction of return variance does OLS explain?
r2_train_ols = r2_score(y_train, y_pred_train_ols)
r2_test_ols  = r2_score(y_test,  y_pred_test_ols)

# RMSE — root mean squared prediction error
rmse_test_ols = np.sqrt(mean_squared_error(y_test, y_pred_test_ols))

print("OLS (Ordinary Least Squares) Results")
print("-" * 40)
print(f"  R² train : {r2_train_ols:.3f}")
print(f"  R² test  : {r2_test_ols:.3f}")
print(f"  RMSE test: {rmse_test_ols:.3f} %")

In [ ]:
# =============================================================================
# SECTION 4, CELL 2: OLS coefficient table — top 5 features by absolute value
# =============================================================================

ols_coefficients = pd.DataFrame({
    'Feature':     X_features.columns.tolist(),
    'OLS_Coef':    model_ols.coef_
})

ols_coefficients['Abs_Coef'] = ols_coefficients['OLS_Coef'].abs()
ols_top5 = ols_coefficients.sort_values('Abs_Coef', ascending=False).head(5)

print("Top 5 OLS features by absolute coefficient (standardized):")
print("-" * 45)
for _, row in ols_top5.iterrows():
    direction = '+' if row['OLS_Coef'] > 0 else '-'
    print(f"  {row['Feature']:<25}  {direction}{abs(row['OLS_Coef']):.4f}")

**Interpretation Checkpoint — OLS Baseline**

- OLS R²_train and R²_test should be close to each other — that is the key diagnostic. When train ≈ test, the model is generalizing rather than memorizing.
- Predicting monthly stock returns is notoriously hard. An R² of 0.10–0.30 in finance is actually meaningful — much of the variance in monthly returns is genuine noise that no model can capture.
- The coefficient table shows which factors OLS associates most strongly with REIT returns. Because we standardized the features, coefficients are on the same scale and can be compared directly.
- This OLS result is our **baseline**. Every other model must beat it on the test set to justify its added complexity.

---
## Section 5: Ridge and Lasso — Regularized Regression

In [ ]:
# =============================================================================
# SECTION 5, CELL 1: Fit Ridge and Lasso with cross-validated alpha selection
# =============================================================================

# --- Ridge Regression ---
# Ridge adds a penalty = alpha * sum(coef²)  — shrinks all coefficients toward zero
alphas_to_try = [0.01, 0.1, 1, 10, 100]

ridge_cv = GridSearchCV(
    Ridge(),
    param_grid={'alpha': alphas_to_try},
    cv=5,
    scoring='r2'
)
ridge_cv.fit(X_train_scaled, y_train)

best_alpha_ridge = ridge_cv.best_params_['alpha']
model_ridge = ridge_cv.best_estimator_

y_pred_train_ridge = model_ridge.predict(X_train_scaled)
y_pred_test_ridge  = model_ridge.predict(X_test_scaled)

r2_train_ridge = r2_score(y_train, y_pred_train_ridge)
r2_test_ridge  = r2_score(y_test,  y_pred_test_ridge)
rmse_test_ridge = np.sqrt(mean_squared_error(y_test, y_pred_test_ridge))

print("Ridge Regression Results")
print("-" * 40)
print(f"  Best alpha (CV): {best_alpha_ridge}")
print(f"  R² train       : {r2_train_ridge:.3f}")
print(f"  R² test        : {r2_test_ridge:.3f}")
print(f"  RMSE test      : {rmse_test_ridge:.3f} %")

print()

# --- Lasso Regression ---
# Lasso adds a penalty = alpha * sum(|coef|)  — drives some coefficients exactly to zero
lasso_cv = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid={'alpha': alphas_to_try},
    cv=5,
    scoring='r2'
)
lasso_cv.fit(X_train_scaled, y_train)

best_alpha_lasso = lasso_cv.best_params_['alpha']
model_lasso = lasso_cv.best_estimator_

y_pred_train_lasso = model_lasso.predict(X_train_scaled)
y_pred_test_lasso  = model_lasso.predict(X_test_scaled)

r2_train_lasso = r2_score(y_train, y_pred_train_lasso)
r2_test_lasso  = r2_score(y_test,  y_pred_test_lasso)
rmse_test_lasso = np.sqrt(mean_squared_error(y_test, y_pred_test_lasso))

n_zero_lasso = np.sum(model_lasso.coef_ == 0)
n_features_total = len(model_lasso.coef_)

print("Lasso Regression Results")
print("-" * 40)
print(f"  Best alpha (CV)          : {best_alpha_lasso}")
print(f"  Coefficients zeroed out  : {n_zero_lasso} of {n_features_total}")
print(f"  R² train                 : {r2_train_lasso:.3f}")
print(f"  R² test                  : {r2_test_lasso:.3f}")
print(f"  RMSE test                : {rmse_test_lasso:.3f} %")

**How Does Regularization Work?**

Both Ridge and Lasso add a **penalty term** to the OLS loss function that punishes large coefficients:

- **Ridge** penalty: α × Σ(βᵢ²) — shrinks all coefficients *toward* zero, but never exactly to zero. Good when many features have small predictive value.
- **Lasso** penalty: α × Σ|βᵢ| — drives some coefficients *exactly* to zero. This is automatic **feature selection**: Lasso tells you "these predictors are redundant."

The hyperparameter **α** controls how strong the penalty is. Too small → barely different from OLS. Too large → all coefficients shrink to zero (underfitting). **Cross-validation** (GridSearchCV, cv=5) finds the α that performs best on held-out folds *within* the training set — so the test set remains untouched.

In [ ]:
# =============================================================================
# SECTION 5, CELL 3: Bar chart — R² train vs test for OLS, Ridge, Lasso
# =============================================================================

linear_models = ['OLS', 'Ridge', 'Lasso']
r2_train_linear = [r2_train_ols, r2_train_ridge, r2_train_lasso]
r2_test_linear  = [r2_test_ols,  r2_test_ridge,  r2_test_lasso]

x_pos = np.arange(len(linear_models))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

bars_train = ax.bar(x_pos - bar_width/2, r2_train_linear, bar_width,
                    label='R² Train', alpha=0.85,
                    color=[MODEL_COLORS[m] for m in linear_models])
bars_test  = ax.bar(x_pos + bar_width/2, r2_test_linear,  bar_width,
                    label='R² Test',  alpha=0.45,
                    color=[MODEL_COLORS[m] for m in linear_models])

# Label each bar with its value
for bar in bars_train:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)
for bar in bars_test:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(linear_models, fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title('Linear Models: R² Train vs. R² Test\n(Closer together = less overfitting)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(r2_train_linear + r2_test_linear) * 1.25)
plt.tight_layout()
plt.show()

print("All three linear models show similar train vs. test R² — they generalize well.")

---
## Section 6: Random Forest — Where Overfitting Gets Dramatic

In [ ]:
# =============================================================================
# SECTION 6, CELL 1: Fit Random Forest
# =============================================================================

# Random Forest: 100 decision trees, each trained on a random subset of features
# max_depth=10 limits tree depth slightly, but still allows significant overfitting
model_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_SEED
)
model_rf.fit(X_train_scaled, y_train)

y_pred_train_rf = model_rf.predict(X_train_scaled)
y_pred_test_rf  = model_rf.predict(X_test_scaled)

r2_train_rf  = r2_score(y_train, y_pred_train_rf)
r2_test_rf   = r2_score(y_test,  y_pred_test_rf)
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))

print("Random Forest Results")
print("-" * 40)
print(f"  R² train : {r2_train_rf:.3f}   <-- suspiciously high")
print(f"  R² test  : {r2_test_rf:.3f}   <-- much lower")
print(f"  RMSE test: {rmse_test_rf:.3f} %")
print()
print(f"  Overfit gap (train - test): {r2_train_rf - r2_test_rf:.3f}")
print("  --> A large gap means the model memorized training data patterns that don't hold out-of-sample.")

**Interpretation Checkpoint — Random Forest Overfitting Signal**

The key number here is the **overfit gap**: R²_train minus R²_test.

- For OLS/Ridge/Lasso, this gap should be small (roughly 0.00–0.05) — these models have a hard constraint (linearity) that prevents them from memorizing noise.
- For Random Forest, the gap is much larger. The forest can grow deep trees that carve out tiny regions of feature space and perfectly predict training observations — but those carved-out patterns are noise artifacts that vanish in new data.

**High training R² is not a sign of a good model. It is a warning sign.** The only R² that matters is the test R².

In [ ]:
# =============================================================================
# SECTION 6, CELL 3: All 4 models — the "Overfitting in Action" chart
# This is the key diagnostic figure of the entire demo.
# =============================================================================

all_models   = ['OLS', 'Ridge', 'Lasso', 'RandomForest']
r2_train_all = [r2_train_ols, r2_train_ridge, r2_train_lasso, r2_train_rf]
r2_test_all  = [r2_test_ols,  r2_test_ridge,  r2_test_lasso,  r2_test_rf]

x_pos = np.arange(len(all_models))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

# Train bars — solid, full opacity
bars_train = ax.bar(x_pos - bar_width/2, r2_train_all, bar_width,
                    label='R² Train (in-sample)',
                    color=[MODEL_COLORS[m] for m in all_models],
                    alpha=0.90, edgecolor='black', linewidth=0.8)

# Test bars — hatched, semi-transparent to make the gap visually obvious
bars_test = ax.bar(x_pos + bar_width/2, r2_test_all, bar_width,
                   label='R² Test  (out-of-sample)',
                   color=[MODEL_COLORS[m] for m in all_models],
                   alpha=0.40, edgecolor='black', linewidth=0.8,
                   hatch='//')

# Annotate each bar
for bar in list(bars_train) + list(bars_test):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2,
            height + 0.01,
            f"{height:.3f}",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Draw a red arrow highlighting the RF overfit gap
rf_idx = 3
ax.annotate(
    f"Overfit Gap\n= {r2_train_rf - r2_test_rf:.3f}",
    xy=(rf_idx + bar_width/2, r2_test_rf),
    xytext=(rf_idx + bar_width/2 + 0.35, (r2_train_rf + r2_test_rf) / 2),
    fontsize=10, color='darkred', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5)
)

ax.set_xticks(x_pos)
ax.set_xticklabels(['OLS', 'Ridge', 'Lasso', 'Random\nForest'], fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title(
    'Overfitting in Action: Random Forest Memorizes, OLS Generalizes\n'
    'Solid = training R²,  Hatched = test R²  (what actually matters)',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=11, loc='upper left')
ax.set_ylim(0, max(r2_train_all) * 1.30)
plt.tight_layout()

# Save figure
fig.savefig('overfit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved as 'overfit_comparison.png'")
print()
print("Key insight: Random Forest has the HIGHEST training R² but NOT the best test R².")
print("OLS and Ridge are boring in-sample but more honest out-of-sample.")

---
## Section 7: Feature Importance — Do All Models Agree?

In [ ]:
# =============================================================================
# SECTION 7, CELL 1: Extract feature importances for all 4 models
# =============================================================================

feature_names = list(X_features.columns)

# OLS: absolute value of standardized coefficients
importance_ols = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': np.abs(model_ols.coef_)
}).sort_values('Importance', ascending=True).tail(7)

# Ridge: absolute value of standardized coefficients
importance_ridge = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': np.abs(model_ridge.coef_)
}).sort_values('Importance', ascending=True).tail(7)

# Lasso: absolute value of standardized coefficients
importance_lasso = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': np.abs(model_lasso.coef_)
}).sort_values('Importance', ascending=True).tail(7)

# Random Forest: built-in feature_importances_ (mean decrease in impurity)
importance_rf = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': model_rf.feature_importances_
}).sort_values('Importance', ascending=True).tail(7)

print("Top 7 features per model extracted. Plotting 2×2 grid...")

In [ ]:
# =============================================================================
# SECTION 7, CELL 2: 2×2 subplot — feature importance per model
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Feature Importance by Model — Do They Agree?', fontsize=14, fontweight='bold', y=1.01)

models_data = [
    ('OLS',          importance_ols,   MODEL_COLORS['OLS']),
    ('Ridge',        importance_ridge, MODEL_COLORS['Ridge']),
    ('Lasso',        importance_lasso, MODEL_COLORS['Lasso']),
    ('Random Forest', importance_rf,   MODEL_COLORS['RandomForest'])
]

for ax, (model_name, imp_df, color) in zip(axes.flatten(), models_data):
    bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
                   color=color, alpha=0.80, edgecolor='white')
    ax.set_title(model_name, fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Importance (abs. coef or mean impurity decrease)', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

print("Do the top features look similar across models?")
print("Notice that Lasso may have zeroed out some features entirely — those don't appear.")

**Interpretation Checkpoint — Feature Importance Across Models**

- **Agreement across models suggests a genuine signal.** If OLS, Ridge, Lasso, and Random Forest all identify the same top-2 predictors, you can be more confident that those predictors genuinely associate with REIT returns — the pattern isn't model-specific.
- **Disagreement is a warning.** If Random Forest places high importance on a feature that OLS ignores, it may be picking up a non-linear interaction that only exists in the training data — another form of overfitting.
- **Random Forest importances ≠ causal effects.** The "importance" score only tells you how much a feature improves splits across all trees. It does *not* tell you the direction of the effect, nor whether the effect is causal. For causal questions, use OLS coefficients (with appropriate controls).
- **Lasso's zeroed-out coefficients are an explicit statement:** those features add no incremental predictive power after the penalty is applied.

---
## Section 8: The Investment Decision

### Which model should you actually use?

Random Forest has the highest **training** accuracy. Does that make it the better model for an investment recommendation? Let's think through three practical questions:

**1. Can you explain each prediction to a client?**
OLS gives you: *"Each 1% increase in the value factor is associated with a 0.4% higher REIT return (p=0.01)."* That's an auditable, plain-language claim. Random Forest gives you: *"The model predicted 2.1%."* You cannot explain which inputs drove that prediction without additional interpretability tools — and even then, explanations are post-hoc approximations.

**2. What happens if Random Forest memorized historical noise?**
The COVID crash (2020) and GFC (2008) dominated the training data volatility. If Random Forest learned that extreme factor moves predict extreme returns *in those specific months*, it built in a pattern that may not repeat. OLS is constrained to a single linear relationship — it has less room to memorize noise.

**3. Which model can be audited for regulatory compliance?**
Financial regulators increasingly require **explainability**. Many fund structures require disclosure of the investment rationale. OLS coefficients are directly interpretable. Random Forest is a black box.

**Bottom line:** For the QM 2023 capstone, your job is to **explain relationships** — to answer "why." That is an inference question, not a prediction competition. OLS (and its panel extensions — Fixed Effects, DiD) remain the right tools. Machine learning is a useful comparison point, and worth understanding — but it does not replace econometric models when causation is the goal.

In [ ]:
# =============================================================================
# SECTION 8, CELL 2: Final comparison table — all 4 models
# =============================================================================

results_table = pd.DataFrame({
    'Model':        ['OLS', 'Ridge', 'Lasso', 'Random Forest'],
    'R2_Train':     [r2_train_ols,   r2_train_ridge,  r2_train_lasso,  r2_train_rf],
    'R2_Test':      [r2_test_ols,    r2_test_ridge,   r2_test_lasso,   r2_test_rf],
    'RMSE_Test':    [rmse_test_ols,  rmse_test_ridge,  rmse_test_lasso, rmse_test_rf],
    'Overfit_Gap':  [
        r2_train_ols   - r2_test_ols,
        r2_train_ridge - r2_test_ridge,
        r2_train_lasso - r2_test_lasso,
        r2_train_rf    - r2_test_rf
    ]
})

# Round for display
for col in ['R2_Train', 'R2_Test', 'RMSE_Test', 'Overfit_Gap']:
    results_table[col] = results_table[col].round(3)

print("=" * 65)
print("FINAL MODEL COMPARISON TABLE")
print("=" * 65)
print(results_table.to_string(index=False))
print("=" * 65)
print()
print("Overfit_Gap = R2_Train - R2_Test")
print("  Small gap (~0.00-0.05) = model generalizes well")
print("  Large gap              = model memorized training noise")
print()
print("Best out-of-sample RMSE:")
best_model_idx = results_table['RMSE_Test'].idxmin()
print(f"  {results_table.loc[best_model_idx, 'Model']}  (RMSE = {results_table.loc[best_model_idx, 'RMSE_Test']:.3f} %)")

---
## Section 9: Key Takeaways

1. **Bias-variance tradeoff:** Complex models (Random Forest) fit training data better but generalize worse. Simple models (OLS) accept higher bias in exchange for lower variance — they make fewer extreme mistakes on new data.

2. **The train-test gap is your overfitting diagnostic.** A small R²_train vs. R²_test gap means the model learned a real pattern. A large gap (especially for Random Forest) means it memorized noise. Always report both.

3. **OLS is "worse" by training accuracy but safer for investment decisions.** It is interpretable, auditable, and its coefficients have a direct meaning. High training R² is not an endorsement — it may simply mean the model cheated.

4. **Ridge shrinks all coefficients; Lasso zeroes some out.** Lasso performs automatic feature selection, telling you which predictors have no marginal predictive value after regularization. The best penalty strength (α) is chosen by cross-validation — never by eyeballing.

5. **Feature importance from Random Forest ≠ causal effect.** It measures how much each feature improves prediction splits on average across trees. It does not tell you the direction of the effect, whether the effect is linear, or whether it would hold in a new market environment.

6. **For the QM 2023 capstone:** Econometric models (OLS, Fixed Effects, Difference-in-Differences) remain the tool for causal questions. Machine learning is a useful forecasting benchmark, but it does not replace causal identification. Knowing the difference — and being able to defend your choice — is the mark of a rigorous analyst.